In [ ]:
# ======================= PDF → VECTORS → ELASTIC =====================
# This cell will:
#  - install dependencies (safe to re-run)
#  - connect to Elasticsearch
#  - create an index with a dense_vector field (if it doesn't exist yet)
#  - read all PDFs from the folders in PDF_DIRS
#  - do paragraph-aware semantic chunking with overlap
#  - create multilingual embeddings on CPU and index them
#  - PRINT PROGRESS: which book, which page, how many chunks, totals

# ---------- install deps (idempotent) ----------------------------------------
!pip install -q "elasticsearch>=8.0.0" sentence-transformers pypdf

# ---------- CONFIG: EDIT THESE VALUES ----------------------------------------

ES_URL       = "http://localhost:9200"        # Elasticsearch as seen from EC2
ES_USER      = "elastic"                      # your ES username
ES_PASSWORD  = "your-password"                    # <<< CHANGE THIS IF NEEDED

INDEX_NAME   = "radio_luxembourg_books"       # index name

# Project root (used to build relative paths for Jupyter links)
PROJECT_ROOT = "/home/ec2-user/rag-project"

# Folders where your PDFs live on the EC2 (inside Jupyter)
PDF_DIRS = [
    # "/home/ec2-user/rag-project/radio_luxembourg",
    "/home/ec2-user/rag-project/radio-luxembourg-in-Poland",
]

# Chunking parameters
MAX_CHARS      = 1200    # target chunk length (characters)
OVERLAP_CHARS  = 200     # overlapping chars between neighbouring chunks

# Multilingual embedding model (FR/DE/LB), CPU-only
MODEL_NAME   = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
EMB_BATCH    = 8  # embedding batch size (lower = less RAM, slower)

# ---------- IMPORTS -----------------------------------------------------------

import os
import glob
import re
from typing import List, Iterable, Tuple
from time import time

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# ---------- CONNECT TO ELASTICSEARCH -----------------------------------------

es = Elasticsearch(ES_URL, basic_auth=(ES_USER, ES_PASSWORD))
info = es.info().body
print("Connected to Elasticsearch cluster:", info.get("cluster_name", info))

# ---------- LOAD EMBEDDING MODEL (CPU) ----------------------------------------

print(f"\nLoading embedding model on CPU: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME, device="cpu")
EMBED_DIMS = model.get_sentence_embedding_dimension()
print("Embedding dimension:", EMBED_DIMS)

# ---------- CREATE INDEX (IF NEEDED) -----------------------------------------

mapping = {
    "mappings": {
        "properties": {
            "book_title":   {"type": "keyword"},
            "file_path":    {"type": "keyword"},   # full path on disk
            "pdf_rel_path": {"type": "keyword"},   # e.g. "radio_luxembourg/Radio Luxembourg 1933-1993.pdf"
            "page_number":  {"type": "integer"},
            "chunk_id":     {"type": "integer"},
            "chunk_text":   {"type": "text"},
            "language":     {"type": "keyword"},   # 'de','fr','lb','multi', ...
            "embedding": {
                "type": "dense_vector",
                "dims": EMBED_DIMS,
                "index": True,
                "similarity": "cosine"
            }
        }
    }
}

if not es.indices.exists(index=INDEX_NAME):
    es.indices.create(index=INDEX_NAME, body=mapping)
    print(f"\nIndex '{INDEX_NAME}' created.")
else:
    print(f"\nIndex '{INDEX_NAME}' already exists – using existing mapping (no data will be deleted).")

# ---------- SEMANTIC-ISH CHUNKING --------------------------------------------

def split_into_paragraphs(text: str) -> List[str]:
    """Split raw page text into paragraphs separated by blank lines."""
    text = text.replace("\r\n", "\n")
    paras = re.split(r"\n{2,}", text)
    return [p.strip() for p in paras if p.strip()]

def make_semantic_chunks_for_page(
    text: str,
    max_chars: int = MAX_CHARS,
    overlap_chars: int = OVERLAP_CHARS,
) -> List[str]:
    """
    Paragraph-aware chunking within a single page.
    Keeps paragraphs together until max_chars is reached.
    When splitting, creates neighbour overlap for smoother semantics.
    """
    paragraphs = split_into_paragraphs(text)
    raw_chunks: List[str] = []
    current = ""

    for p in paragraphs:
        if not current:
            current = p
            continue

        if len(current) + len(p) + 2 <= max_chars:
            current = current + "\n\n" + p
        else:
            raw_chunks.append(current)

            # Very long paragraph → slice with overlap
            if len(p) > max_chars:
                start = 0
                while start < len(p):
                    end = start + max_chars
                    raw_chunks.append(p[start:end])
                    start = end - overlap_chars
                current = ""
            else:
                current = p

    if current:
        raw_chunks.append(current)

    # Add overlap between neighbouring chunks on the same page
    chunks: List[str] = []
    for i, chunk in enumerate(raw_chunks):
        if i == 0:
            chunks.append(chunk)
        else:
            overlap = raw_chunks[i-1][-overlap_chars:]
            chunks.append(overlap + "\n\n" + chunk)

    return chunks

# ---------- PDF TEXT EXTRACTION (PER PAGE) -----------------------------------

def extract_pages_from_pdf(pdf_path: str) -> List[str]:
    """Return list of page texts (index 0 = page 1)."""
    reader = PdfReader(pdf_path)
    pages = []
    for idx, page in enumerate(reader.pages):
        try:
            pages.append(page.extract_text() or "")
        except Exception as e:
            print(f"    [WARN] Failed to extract text from page {idx+1}: {e}")
            pages.append("")
    return pages

# ---------- SIMPLE LANGUAGE TAG FROM FILENAME --------------------------------

def guess_language_from_filename(filename: str) -> str:
    fn = filename.lower()
    if "deutsch" in fn or "german" in fn or fn.endswith("_de.pdf"):
        return "de"
    if "francais" in fn or "français" in fn or fn.endswith("_fr.pdf"):
        return "fr"
    # extend with your own rules if file names encode language
    return "multi"  # unknown / mixed

# ---------- GENERATOR OF BULK ACTIONS ----------------------------------------

def iter_es_actions() -> Iterable[dict]:
    # Collect PDFs from all configured folders
    pdf_files: List[str] = []
    for folder in PDF_DIRS:
        pdf_files.extend(glob.glob(os.path.join(folder, "*.pdf")))

    pdf_files = sorted(pdf_files)
    total_pdfs = len(pdf_files)

    if not pdf_files:
        print("No PDF files found in any of:", PDF_DIRS)
        return

    global_doc_counter = 0

    for file_idx, pdf_path in enumerate(pdf_files):
        file_name = os.path.basename(pdf_path)
        print(f"\n[{file_idx+1}/{total_pdfs}] Processing file: {file_name}")
        print(f"    Path: {pdf_path}")

        pages = extract_pages_from_pdf(pdf_path)
        num_pages = len(pages)
        language = guess_language_from_filename(file_name)
        print(f"    Detected language tag: {language}")
        print(f"    Total pages detected: {num_pages}")

        all_chunks: List[str] = []
        all_meta: List[Tuple[int, int]] = []  # (page_number, chunk_id)
        chunk_counter = 0

        for page_idx, page_text in enumerate(pages):
            page_number = page_idx + 1
            if not page_text.strip():
                print(f"    Page {page_number:3d}/{num_pages}: [NO TEXT] – skipping")
                continue

            page_chunks = make_semantic_chunks_for_page(page_text)
            print(f"    Page {page_number:3d}/{num_pages}: {len(page_chunks)} chunk(s)")

            for ch in page_chunks:
                all_chunks.append(ch)
                all_meta.append((page_number, chunk_counter))
                chunk_counter += 1

        if not all_chunks:
            print("    -> No text extracted from this PDF, skipping.")
            continue

        print(f"    Total chunks for this book: {len(all_chunks)}")
        print("    Encoding embeddings on CPU ...")

        embeddings = model.encode(
            all_chunks,
            normalize_embeddings=True,
            convert_to_numpy=True,
            batch_size=EMB_BATCH,
            show_progress_bar=True,
        )

        for (chunk_text, emb_vec), (page_number, chunk_id) in zip(
            zip(all_chunks, embeddings), all_meta
        ):
            # Relative path from PROJECT_ROOT, so Jupyter links work for all folders
            pdf_rel_path = os.path.relpath(pdf_path, PROJECT_ROOT)

            global_doc_counter += 1

            # Deterministic ID → re-running will overwrite same doc, not duplicate
            doc_id = f"{file_name}-p{page_number}-c{chunk_id}"

            # Print a light progress line every 100 docs
            if global_doc_counter % 100 == 0:
                print(
                    f"      -> Prepared {global_doc_counter} chunks so far "
                    f"(book: {file_name}, page: {page_number}, chunk_id: {chunk_id})"
                )

            yield {
                "_index": INDEX_NAME,
                "_id": doc_id,
                "_source": {
                    "book_title":   file_name,
                    "file_path":    pdf_path,
                    "pdf_rel_path": pdf_rel_path,
                    "page_number":  page_number,
                    "chunk_id":     chunk_id,
                    "chunk_text":   chunk_text,
                    "language":     language,
                    "embedding":    emb_vec.tolist(),
                },
            }

    print("\nFinished preparing all chunks for all PDFs.")

# ---------- RUN BULK INDEXING ------------------------------------------------

print("\nStarting bulk indexing into Elasticsearch...")
t0 = time()
success_count, errors = bulk(
    es,
    iter_es_actions(),
    chunk_size=100,
    request_timeout=300,
    raise_on_error=False,
)
t1 = time()

print(f"\nDONE. Successfully indexed documents: {success_count}")
print(f"Took {(t1 - t0):.1f} seconds total.")
if errors:
    print("Some errors occurred (showing first 3):")
    for e in errors[:3]:
        print(e)


/home/ec2-user/rag-project/.venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Elasticsearch cluster: elasticsearch

Loading embedding model on CPU: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Embedding dimension: 768

Index 'radio_luxembourg_books' already exists – using existing mapping (no data will be deleted).

Starting bulk indexing into Elasticsearch...

[1/1] Processing file: Remembering Radio Luxembourg in the Peoples Republic of Poland.pdf
    Path: /home/ec2-user/rag-project/radio-luxembourg-in-Poland/Remembering Radio Luxembourg in the Peoples Republic of Poland.pdf
    Detected language tag: multi
    Total pages detected: 103
    Page   1/103: [NO TEXT] – skipping
    Page   2/103: [NO TEXT] – skipping
    Page   3/103: [NO TEXT] – skipping
    Page   4/103: 1 chunk(s)
    Page   5/103: 1 chunk(s)
    Page   6/103: [NO TEXT] – skipping
    Page   7/103: 1 chunk(s)
    Page   8/103: 1 chunk(s)
    Page   9/103: 1 chunk(s)
    Page  10/103: 1 chunk(s)
    Page  11/103: 1 chunk(s)
    Page  12/103: 1 chunk(s)
    Page  13/1

Batches: 100%|██████████| 12/12 [00:12<00:00,  1.02s/it]
/tmp/ipykernel_245801/1483246030.py:275: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  success_count, errors = bulk(



Finished preparing all chunks for all PDFs.

DONE. Successfully indexed documents: 93
Took 13.1 seconds total.
